<a href="https://colab.research.google.com/github/htranhp1213/RAG-glaucoma/blob/main/02_ingest_and_chunk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Session 1 – Problem, Domain, and Setup

* Defined a real clinical use case (glaucoma decision-support) with a clear target user and impact statement aligned to healthcare AI roles.

* Curated ~10 authoritative, publicly available glaucoma guideline documents (AAO, NICE, EGS, institutional protocols).

* Built an initial Colab pipeline to load PDFs and verify clean, readable text extraction.

In [4]:
# Session 1: Load and inspect documents
# Goal: Load PDFs and print cleaned text

!pip install pypdf

from pypdf import PdfReader
from pathlib import Path
import re

DATA_DIR = Path("/content")

def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)   # collapse whitespace
    return text.strip()

def load_pdf(path: Path) -> str:
    reader = PdfReader(path)
    pages = []
    for page in reader.pages:
        if page.extract_text():
            pages.append(page.extract_text())
    return "\n".join(pages)

# Load at least 3 documents
pdf_files = list(DATA_DIR.glob("*.pdf"))[:3]

for pdf_path in pdf_files:
    print("=" * 80)
    print(f"DOCUMENT: {pdf_path.name}")
    print("=" * 80)

    raw_text = load_pdf(pdf_path)
    cleaned_text = clean_text(raw_text)

    # Print only first ~1000 characters to keep output readable
    print(cleaned_text[:1000])
    print("\n\n")


DOCUMENT: 2022_Measure_141_MIPSCQM.pdf
Version 6.0 CPT only copyright 2021 American Medical Association. All rights reserved. December 2021 Page 1 of 8 Quality ID #141 (NQF 0563): Primary Open-Angle Glaucoma (POAG): Reduction of Intraocular Pressure (IOP) by 15% OR Documentation of a Plan of Care – National Quality Strategy Domain: Communication and Care Coordination – Meaningful Measure Area: Management of Chronic Conditions 2022 COLLECTION TYPE: MIPS CLINICAL QUALITY MEASURES (CQMS) MEASURE TYPE: Outcome – High Priority DESCRIPTION: Percentage of patients aged 18 years and older with a diagnosis of primary open-angle glaucoma (POAG) whose glaucoma treatment has not failed (the most recent IOP was reduced by at least 15% from the pre-intervention level) OR if the most recent IOP was not reduced by at least 15% from the pre-intervention level, a plan of care was documented within the 12 month performance period. INSTRUCTIONS: This measure is to be submitted a minimum of once per perfor

Session 2 – Ingestion & Chunking

* Implemented a robust ingestion pipeline to load, clean, and normalize clinical guideline PDFs with consistent metadata.

* Applied recursive text chunking (600 chars, 100 overlap) optimized for medical documents, preserving semantic boundaries.

* Generated and inspected chunk statistics (total chunks, average length, per-document distribution) to validate retrieval readiness.





In [5]:
# Session 2: Ingestion chunking
# Goal: Split into chunks

# Double check PDFs
!pip -q install pypdf langchain-community langchain-text-splitters

from pathlib import Path

DATA_DIR = Path("/content")   # change to Path("/content/data") if you have a data folder
pdf_paths = sorted(DATA_DIR.glob("*.pdf"))

print("Found PDFs:", len(pdf_paths))
for p in pdf_paths[:10]:
    print("-", p.name)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
Found PDFs: 10
- 1.Primary Open-Angle Glaucoma PPP.pdf
- 2.glaucoma-diagnosis-and-management-pdf-1837689655237.pdf
- 2022_Measure_141_MIPSCQM.pdf
- 3.ICOGlaucomaGuidelines.pdf
- CPG.VP.30-Glaucoma.pdf
- EGS_Guidelines_5_English-1.pdf
- EHCO-Glaucoma-Guidelines.27Feb2013.Final_.pdf
- RANZCO-Clinical-Practice-Guidelines-for-the-Collaborative-Care-of-glaucoma-patients_2.pdf
- cdc_131539_DS1.pdf
- glaucoma_diagosis.pd

In [6]:
# Load PDFs with page metadata (LangChain loader)
from langchain_community.document_loaders import PyPDFLoader

docs = []
for p in pdf_paths:
    loader = PyPDFLoader(str(p))
    docs.extend(loader.load())

print("Loaded pages:", len(docs))
print("Example metadata:", docs[0].metadata)
print("Example text preview:", docs[0].page_content[:300])


Loaded pages: 387
Example metadata: {'producer': 'Acrobat Distiller 6.0 for Windows', 'creator': 'Elsevier', 'creationdate': '2020-12-12T19:54:03+05:30', 'crossmarkdomains[2]': 'sciencedirect.com', 'crossmarkmajorversiondate': '2010-04-23', 'subject': 'Ophthalmology, 128 (2020) P71-P150. doi:10.1016/j.ophtha.2020.10.022', 'author': 'Steven J. Gedde MD', 'keywords': '', 'elsevierwebpdfspecifications': '7.0', 'crossmarkdomainexclusive': 'true', 'robots': 'noindex', 'moddate': '2020-12-24T16:35:56+05:30', 'doi': '10.1016/j.ophtha.2020.10.022', 'crossmarkdomains[1]': 'elsevier.com', 'title': 'Primary Open-Angle Glaucoma Preferred Practice Pattern®', 'source': '/content/1.Primary Open-Angle Glaucoma PPP.pdf', 'total_pages': 80, 'page': 0, 'page_label': 'P71'}
Example text preview: Primary Open-Angle 
Glaucoma Preferred 
Practice Pattern®
P71


In [7]:
# Clean text, normalize whitespace
import re

def clean_text(t: str) -> str:
    t = re.sub(r"\s+", " ", t)  # collapse whitespace
    return t.strip()

for d in docs:
    d.page_content = clean_text(d.page_content)


In [16]:
# Derive doc_name from source

from pathlib import Path

for d in docs:
    src = d.metadata.get("source", "")
    d.metadata["doc_name"] = Path(src).name if src else "unknown.pdf"
    d.metadata["source_type"] = "pdf"

    # normalize page label
    if "page" in d.metadata:
        d.metadata["page_label"] = int(d.metadata["page"]) + 1


In [17]:
# Chunk with RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(docs)

print("Total chunks:", len(chunks))
print("Sample chunk metadata:", chunks[0].metadata)
print("Sample chunk text:", chunks[0].page_content[:400])


Total chunks: 2029
Sample chunk metadata: {'producer': 'Acrobat Distiller 6.0 for Windows', 'creator': 'Elsevier', 'creationdate': '2020-12-12T19:54:03+05:30', 'crossmarkdomains[2]': 'sciencedirect.com', 'crossmarkmajorversiondate': '2010-04-23', 'subject': 'Ophthalmology, 128 (2020) P71-P150. doi:10.1016/j.ophtha.2020.10.022', 'author': 'Steven J. Gedde MD', 'keywords': '', 'elsevierwebpdfspecifications': '7.0', 'crossmarkdomainexclusive': 'true', 'robots': 'noindex', 'moddate': '2020-12-24T16:35:56+05:30', 'doi': '10.1016/j.ophtha.2020.10.022', 'crossmarkdomains[1]': 'elsevier.com', 'title': 'Primary Open-Angle Glaucoma Preferred Practice Pattern®', 'source': '/content/1.Primary Open-Angle Glaucoma PPP.pdf', 'total_pages': 80, 'page': 0, 'page_label': 1, 'doc_name': '1.Primary Open-Angle Glaucoma PPP.pdf', 'source_type': 'pdf'}
Sample chunk text: Primary Open-Angle Glaucoma Preferred Practice Pattern® P71


In [18]:
# Inspect 3-5 sample
for i in range(5):
    print("="*90)
    print(f"CHUNK {i}")
    print("doc:", chunks[i].metadata.get("doc_name"), "| page:", chunks[i].metadata.get("page_label"))
    print(chunks[i].page_content[:800])


CHUNK 0
doc: 1.Primary Open-Angle Glaucoma PPP.pdf | page: 1
Primary Open-Angle Glaucoma Preferred Practice Pattern® P71
CHUNK 1
doc: 1.Primary Open-Angle Glaucoma PPP.pdf | page: 2
Secretary for Quality of Care Timothy W. Olsen, MD Academy Staff Ali Al-Rajhi, PhD, MPH Andre Ambrus, MLIS Meghan Daly Flora C. Lum, MD Medical Editor: Susan Garratt Approved by: Board of Trustees September 12, 2020 Copyright © 2020 American Academy of Ophthalmology® All rights reserved AMERICAN ACADEMY OF OPHTHALMOLOGY and PREFERRED PRACTICE PATTERN are registered trademarks of the American Academy of Ophthalmology. All other trademarks are the property of their respective owners. Preferred Practice Pattern® guidelines are developed by the Academy’s H
CHUNK 2
doc: 1.Primary Open-Angle Glaucoma PPP.pdf | page: 2
. Preferred Practice Pattern® guidelines are developed by the Academy’s H. Dunbar Hoskins Jr., MD Center for Quality Eye Care without any external financial support. Authors and reviewers of the gui

In [19]:
# Compute stats
from collections import Counter
import numpy as np

lengths = [len(c.page_content) for c in chunks]
chunks_per_doc = Counter([c.metadata.get("doc_name","unknown") for c in chunks])

print("----- STATS -----")
print("Total docs (PDF files):", len(pdf_paths))
print("Total pages loaded:", len(docs))
print("Total chunks:", len(chunks))
print("Avg chunk length (chars):", round(float(np.mean(lengths)), 2))
print("Min/Max chunk length:", min(lengths), "/", max(lengths))

print("\nChunks per document:")
for name, cnt in chunks_per_doc.most_common():
    print(f"- {name}: {cnt}")


----- STATS -----
Total docs (PDF files): 10
Total pages loaded: 387
Total chunks: 2029
Avg chunk length (chars): 467.3
Min/Max chunk length: 2 / 600

Chunks per document:
- 1.Primary Open-Angle Glaucoma PPP.pdf: 673
- EGS_Guidelines_5_English-1.pdf: 638
- cdc_131539_DS1.pdf: 163
- 2.glaucoma-diagnosis-and-management-pdf-1837689655237.pdf: 147
- glaucoma_diagosis.pdf: 104
- RANZCO-Clinical-Practice-Guidelines-for-the-Collaborative-Care-of-glaucoma-patients_2.pdf: 92
- 3.ICOGlaucomaGuidelines.pdf: 77
- 2022_Measure_141_MIPSCQM.pdf: 51
- EHCO-Glaucoma-Guidelines.27Feb2013.Final_.pdf: 44
- CPG.VP.30-Glaucoma.pdf: 40


In [20]:
# Save chunks
import json

OUT_PATH = Path("/content/chunks.jsonl")

with OUT_PATH.open("w", encoding="utf-8") as f:
    for c in chunks:
        row = {
            "text": c.page_content,
            "metadata": c.metadata
        }
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Saved:", OUT_PATH)


Saved: /content/chunks.jsonl


In [27]:
# CLone gitHub repo
!git clone https://github.com/htranhp1213/RAG-glaucoma.git
%cd RAG-glaucoma


Cloning into 'RAG-glaucoma'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 13 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 14.29 MiB | 16.89 MiB/s, done.
/content/RAG-Project-Glaucoma/RAG-glaucoma/RAG-glaucoma


In [ ]:
## This is for demo purpose only
!pip -q install langchain-openai

import os
os.environ["OPENAI_API_KEY"] = ""

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")



In [ ]:
## This is for demo purpose only
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
"""You must answer using ONLY the context below.
Cite sources as (doc_name, page_label). If context is insufficient, say "I don't know".

Question: {question}

Context:
{context}
"""
)

def rag_answer(question: str):
    docs = retriever.get_relevant_documents(question)
    context = "\n\n".join(
        [f"[{d.metadata.get('doc_name')} p.{d.metadata.get('page_label')}] {d.page_content}" for d in docs]
    )
    msg = prompt.format_messages(question=question, context=context)
    resp = llm.invoke(msg)
    return resp.content

print(rag_answer("How often should POAG patients be monitored?"))
